In [1]:
import pandas as pd
import numpy as np

In [2]:
dataFrame=pd.read_csv('D:/Urban-Traffic-and-Parking-Analysis-using-LSTM-Autoencoders-and-Reinforcement-Learning/Dataset/big_data.csv')

In [3]:
dataFrame.columns

Index(['u', 'v', 'u_lat', 'u_lon', 'v_lat', 'v_lon', 'length_m', 'timestamp',
       'hour', 'day_of_week', 'current_speed', 'free_flow_speed', 'congestion',
       'travel_time_sec', 'rain', 'incident', 'hour_sine', 'hour_cos',
       'day_sin', 'day_cos', 'speed_ratio'],
      dtype='str')

In [4]:
FEATURE_COLS = [
    "hour_sine", "hour_cos",
    "day_sin", "day_cos",
    "travel_time_sec",
    "rain", "incident",
    "congestion",
    "length_m",
    "speed_ratio"
]
TARGET_COL = "speed_ratio"

In [5]:
def sequence_generator(df, seq_len=6, pred_len=3, batch_size=1024):

    FEATURE_COLS = [
        "hour_sine", "hour_cos",
        "day_sin", "day_cos",
        "travel_time_sec",
        "rain", "incident",
        "congestion",
        "length_m",
        "speed_ratio"
    ]

    TARGET_COL = "speed_ratio"

    X_batch = []
    y_batch = []

    current_count = 0

    # group by road
    grouped = df.groupby(["u", "v"])

    for (u, v), group in grouped:

        # sort time (IMPORTANT)
        group = group.sort_values("timestamp")

        data = group[FEATURE_COLS].values
        target = group[TARGET_COL].values

        N = len(group)

        # skip small roads
        if N < seq_len + pred_len:
            continue

        # sliding window
        for i in range(N - seq_len - pred_len + 1):

            X_seq = data[i : i + seq_len]                 # (6, 10)
            y_seq = target[i + seq_len : i + seq_len + pred_len]  # (3,)

            X_batch.append(X_seq)
            y_batch.append(y_seq)

            current_count += 1

            # yield batch
            if current_count == batch_size:

                yield np.array(X_batch, dtype=np.float32), np.array(y_batch, dtype=np.float32)

                X_batch = []
                y_batch = []
                current_count = 0

    # last batch
    if current_count > 0:
        yield np.array(X_batch, dtype=np.float32), np.array(y_batch, dtype=np.float32)

In [6]:
gen = sequence_generator(dataFrame, seq_len=6, pred_len=3, batch_size=1024)

In [71]:
i=0
for X, y in gen:
    i+=1
    # print(X.shape)  # (batch, 6, 10)
    # print(y.shape)  # (batch, 3)
print(i)

22491


In [17]:
import numpy as np

# Step 1: get sorted timestamps
timestamps = np.array(sorted(dataFrame["timestamp"].unique()))

# Step 2: define block size
BLOCK_SIZE = 6   # hours (IMPORTANT: >= seq_len)

# Step 3: split into blocks
blocks = [timestamps[i:i+BLOCK_SIZE] for i in range(0, len(timestamps), BLOCK_SIZE)]

train_times = []
val_times = []
test_times = []

# Step 4: assign blocks
for i, block in enumerate(blocks):

    if i % 5 in [0, 1, 2]:   # 60% train
        train_times.extend(block)

    elif i % 5 == 3:         # 20% val
        val_times.extend(block)

    else:                    # 20% test
        test_times.extend(block)

# Convert to set (faster lookup)
train_times = set(train_times)
val_times   = set(val_times)
test_times  = set(test_times)

# Step 5: filter dataframe
train_df = dataFrame[dataFrame["timestamp"].isin(train_times)]
val_df   = dataFrame[dataFrame["timestamp"].isin(val_times)]
test_df  = dataFrame[dataFrame["timestamp"].isin(test_times)]

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

Train size: 15542172
Val size: 4317270
Test size: 4317270


In [51]:
train_df['day_of_week'].unique()

array([0, 1, 2, 3, 4, 5, 6])

In [19]:
test_df['day_of_week'].unique()

array([1, 2, 3, 4, 6])

In [20]:
val_df['day_of_week'].unique()

array([0, 2, 3, 4, 5])

In [52]:
train_df.head()

,u,v,u_lat,u_lon,v_lat,v_lon,length_m,timestamp,hour,day_of_week,...,free_flow_speed,congestion,travel_time_sec,rain,incident,hour_sine,hour_cos,day_sin,day_cos,speed_ratio
0,30610511.0,1.541914e+09,19.193,73.096,19.193,73.096,-0.679149,2024-01-01 00:00:00,0,0,...,40.0,0.594,-0.667605,0,0,0.000000,1.000000,0.0,1.0,0.406125
1,30610511.0,1.541914e+09,19.193,73.096,19.193,73.096,-0.679149,2024-01-01 01:00:00,1,0,...,40.0,0.626,-0.663681,0,0,0.258819,0.965926,0.0,1.0,0.374175
2,30610511.0,1.541914e+09,19.193,73.096,19.193,73.096,-0.679149,2024-01-01 02:00:00,2,0,...,40.0,0.668,-0.657381,0,0,0.500000,0.866025,0.0,1.0,0.332225
3,30610511.0,1.541914e+09,19.193,73.096,19.193,73.096,-0.679149,2024-01-01 03:00:00,3,0,...,40.0,0.641,-0.661574,0,0,0.707107,0.707107,0.0,1.0,0.358900
4,30610511.0,1.541914e+09,19.193,73.096,19.193,73.096,-0.679149,2024-01-01 04:00:00,4,0,...,40.0,0.615,-0.665049,0,0,0.866025,0.500000,0.0,1.0,0.384575


In [54]:
train_gen = sequence_generator(train_df, seq_len=6, pred_len=3, batch_size=1024)
test_gen = sequence_generator(test_df, seq_len=6, pred_len=3, batch_size=1024)
val_gen = sequence_generator(val_df, seq_len=6, pred_len=3, batch_size=1024)

In [59]:
import torch
import torch.nn as nn

class AutoEncoder(nn.Module):
    def __init__(self, input_dim=10):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 8)   # latent
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

In [60]:
criterion_ae = nn.MSELoss()

In [61]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=128,
            num_layers=3,
            dropout=0.2,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 3)  
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [62]:
criterion_lstm = nn.MSELoss()

In [64]:
def train_autoencoder(model, generator, epochs=9, device="cuda" if torch.cuda.is_available() else "cpu"):

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        count = 0

        for X, _ in generator:

            X = torch.tensor(X.reshape(-1, 10), dtype=torch.float32).to(device)

            optimizer.zero_grad()

            recon, _ = model(X)
            loss = criterion_ae(recon, X)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            count += 1

        avg_loss = epoch_loss / count
        losses.append(avg_loss)

        print(f"AE Epoch {epoch+1} Loss: {avg_loss:.4f}")

    return losses

In [65]:
def encode_batch(model, X, device="cuda"):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    _, z = model(X)
    return z.detach().cpu().numpy()

In [66]:
def train_lstm(model, ae_model, generator, epochs=11, device="cuda" if torch.cuda.is_available() else "cpu"):

    model.to(device)
    ae_model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        count = 0

        for X, y in generator:

            # encode each timestep
            B, T, F = X.shape

            X_flat = X.reshape(-1, F)
            Z = encode_batch(ae_model, X_flat, device)
            Z = Z.reshape(B, T, -1)

            X_tensor = torch.tensor(Z, dtype=torch.float32).to(device)
            y_tensor = torch.tensor(y, dtype=torch.float32).to(device)

            optimizer.zero_grad()

            pred = model(X_tensor)
            loss = criterion_lstm(pred, y_tensor)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            count += 1

        avg_loss = epoch_loss / count
        losses.append(avg_loss)

        print(f"LSTM Epoch {epoch+1} Loss: {avg_loss:.4f}")

    return losses

In [67]:
autoencoder=AutoEncoder()

In [69]:
ae_losses=train_autoencoder(autoencoder,train_gen)

AE Epoch 1 Loss: 0.0435


ZeroDivisionError: division by zero

In [70]:
for X, y in train_gen:
    print(X.shape) 
    print(y.shape)  
    break
for X, y in test_gen:
    print(X.shape) 
    print(y.shape)  
    break
for X, y in val_gen:
    print(X.shape) 
    print(y.shape)  
    break

(1024, 6, 10)
(1024, 3)
(1024, 6, 10)
(1024, 3)
